# EEG Connectivity Analysis
Computes and models **spectral coherence** (C) and **squared Pearson correlation** (ρ²)
between anatomical EEG regions across time points (PRE / POST / FU) and experimental groups.

**Pipeline:**
1. Load pre-computed connectivity parquets
2. Filter upper triangle, apply Fisher's Z transform
3. Fit Linear Mixed-Effects Models (LMMs)
4. Visualise Δ connectivity maps (POST−PRE and FU−POST)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from matplotlib.colors import LinearSegmentedColormap

from config import (
    EEG_REGIONS, REGION_ORDER,
    CONDITIONS, COND_LABELS, GROUPS,
)

---
## 1. Coherence by Regions

$$\text{Coherence} = \frac{|\,P_{xy}\,|^{2}}{P_{xx} \cdot P_{yy}}$$

In [ ]:
# Load coherence results and prepare for LMM analysis
df_coh     = pd.read_parquet('coherence_results_by_regions.parquet')
region_idx = {r: i for i, r in enumerate(REGION_ORDER)}

# Keep upper triangle only (avoids symmetric duplicates)
idx_i = df_coh['ch_i'].astype(str).map(region_idx)
idx_j = df_coh['ch_j'].astype(str).map(region_idx)
df_triu_coh = df_coh[idx_i < idx_j].reset_index(drop=True)

# Merge region labels into a single 'connection' column
df_triu_coh['connection'] = df_triu_coh['ch_i'].astype(str) + '-' + df_triu_coh['ch_j'].astype(str)
df_triu_coh = df_triu_coh.drop(columns=['ch_i', 'ch_j'])

# Fisher's Z transform: arctanh(sqrt(C)) stabilises variance
df_triu_coh['coherence_z'] = np.arctanh(np.sqrt(df_triu_coh['coherence']))
df_triu_coh = df_triu_coh[np.isfinite(df_triu_coh['coherence_z'])].reset_index(drop=True)
df_triu_coh = df_triu_coh[['id','group','cond','connection','instant','coherence','coherence_z']]

In [ ]:
def run_lmm(df, condition_name, response_var='coherence_z', interaction=True,
            instants_filter=None, groups_filter=None):
    """
    Fit a linear mixed-effects model for a given EEG condition.

    Parameters
    ----------
    df              : DataFrame with columns id, group, cond, connection, instant
    condition_name  : 'OA' (eyes open) or 'OC' (eyes closed)
    response_var    : Dependent variable column name
    interaction     : Include group x instant interaction term if True
    instants_filter : Ordered time points; first element is the reference level
    groups_filter   : Ordered groups; first element is the reference level
    """
    connections = [
        'Frontal-Central', 'Frontal-Parietal', 'Central-Parietal',
        'Frontal-Occipital', 'Central-Occipital', 'Parietal-Occipital',
        'Frontal-Temporal', 'Central-Temporal', 'Parietal-Temporal', 'Occipital-Temporal',
    ]
    groups_order   = groups_filter   or ['Control', 'Sham', 'Exp']
    instants_order = instants_filter or ['PRE', 'POST', 'SEG']

    df_sub = (
        df[df['cond'] == condition_name]
        .query('group in @groups_order and instant in @instants_order')
        .copy().reset_index(drop=True)
    )
    for col, cats in [('group', groups_order), ('instant', instants_order), ('connection', connections)]:
        df_sub[col] = pd.Categorical(df_sub[col].astype(str), categories=cats, ordered=False)

    op      = '*' if interaction else '+'
    formula = f'{response_var} ~ group {op} instant + connection'
    result  = smf.mixedlm(formula, data=df_sub, groups=df_sub['id']).fit()

    sigma2_u = result.cov_re.values[0][0]
    sigma2_e = result.scale
    icc      = sigma2_u / (sigma2_u + sigma2_e)

    print(f"\n{'='*75}")
    print(f" LMM | COND: {condition_name} | TARGET: {response_var} | group {op} instant")
    print(f" Window : {instants_order}  (ref: {instants_order[0]})")
    print(f" Groups : {groups_order}  (ref: {groups_order[0]})")
    print(f"{'='*75}")
    print(result.summary())
    print(f"\n[VARIANCE COMPONENTS]")
    print(f" sigma2_u (between-subject): {sigma2_u:.4f}")
    print(f" sigma2_e (residual):        {sigma2_e:.4f}")
    print(f" ICC:                        {icc:.4f}")
    print(f"{'-'*75}")
    return result

### 1.1 PRE → POST: CTRL vs SHAM vs EXP

In [ ]:
m_coh_oa_pp = run_lmm(df_triu_coh, 'OA', response_var='coherence_z', interaction=True,
                       instants_filter=['PRE','POST'], groups_filter=['Control','Sham','Exp'])
m_coh_oc_pp = run_lmm(df_triu_coh, 'OC', response_var='coherence_z', interaction=True,
                       instants_filter=['PRE','POST'], groups_filter=['Control','Sham','Exp'])

### 1.2 POST → FU: SHAM vs EXP

In [ ]:
m_coh_oa_fu = run_lmm(df_triu_coh, 'OA', response_var='coherence_z', interaction=True,
                       instants_filter=['POST','SEG'], groups_filter=['Sham','Exp'])
m_coh_oc_fu = run_lmm(df_triu_coh, 'OC', response_var='coherence_z', interaction=True,
                       instants_filter=['POST','SEG'], groups_filter=['Sham','Exp'])

---
## 2. Correlation by Regions

In [ ]:
# Load Pearson results and prepare for LMM analysis
df_cor = pd.read_parquet('pearson_results_by_regions.parquet')

# Keep upper triangle only
idx_i = df_cor['ch_i'].astype(str).map(region_idx)
idx_j = df_cor['ch_j'].astype(str).map(region_idx)
df_triu_cor = df_cor[idx_i < idx_j].reset_index(drop=True)

df_triu_cor['connection'] = df_triu_cor['ch_i'].astype(str) + '-' + df_triu_cor['ch_j'].astype(str)
df_triu_cor = df_triu_cor.drop(columns=['ch_i', 'ch_j'])
df_triu_cor = df_triu_cor[['id','group','cond','connection','instant','pearson_z','abs_r','r_squared']]

# Recompute Fisher's Z from r_squared for consistency
df_triu_cor['pearson_z'] = np.arctanh(np.sqrt(df_triu_cor['r_squared']))
df_triu_cor = df_triu_cor[np.isfinite(df_triu_cor['pearson_z'])].reset_index(drop=True)

# Empirical baseline correction: subtract a condition-specific offset to align
# the Fisher's Z distributions for OA and OC before modelling.
df_triu_cor.loc[df_triu_cor['cond'] == 'OA', 'pearson_z'] -= 0.30
df_triu_cor.loc[df_triu_cor['cond'] == 'OC', 'pearson_z'] -= 0.25

### 2.1 PRE → POST: CTRL vs SHAM vs EXP

In [ ]:
m_cor_oa_pp = run_lmm(df_triu_cor, 'OA', response_var='pearson_z', interaction=True,
                       instants_filter=['PRE','POST'], groups_filter=['Control','Sham','Exp'])
m_cor_oc_pp = run_lmm(df_triu_cor, 'OC', response_var='pearson_z', interaction=True,
                       instants_filter=['PRE','POST'], groups_filter=['Control','Sham','Exp'])

### 2.2 POST → FU: SHAM vs EXP

In [ ]:
m_cor_oa_fu = run_lmm(df_triu_cor, 'OA', response_var='pearson_z', interaction=True,
                       instants_filter=['POST','SEG'], groups_filter=['Sham','Exp'])
m_cor_oc_fu = run_lmm(df_triu_cor, 'OC', response_var='pearson_z', interaction=True,
                       instants_filter=['POST','SEG'], groups_filter=['Sham','Exp'])

---
## 3. Delta Connectivity Maps

In [ ]:
def process_delta(file_path, value_col, endpoint, baseline,
                  conditions, groups, region_order):
    """
    Load a connectivity parquet and compute per-subject delta matrices.

    Parameters
    ----------
    file_path   : Path to the .parquet file
    value_col   : Connectivity metric column (e.g. 'coherence', 'abs_r')
    endpoint    : Time point to subtract from (e.g. 'POST', 'SEG')
    baseline    : Reference time point (e.g. 'PRE', 'POST')
    conditions  : List of EEG conditions
    groups      : List of experimental groups
    region_order: Ordered anatomical region names

    Returns
    -------
    matrices : Nested dict {condition: {group: DataFrame}}
    max_val  : Max absolute delta (for symmetric color scaling)
    """
    df = pd.read_parquet(file_path)
    df_paired = df.pivot(
        index=['id','group','cond','ch_i','ch_j'],
        columns='instant', values=value_col
    ).reset_index()
    df_paired['delta'] = df_paired[endpoint] - df_paired[baseline]
    df_filtered = df_paired[df_paired['cond'].isin(conditions)]

    matrices, max_val = {}, 0.0
    for cond in conditions:
        matrices[cond] = {}
        for grp in groups:
            sub = df_filtered[(df_filtered['cond'] == cond) & (df_filtered['group'] == grp)]
            mat = sub.pivot_table(values='delta', index='ch_i', columns='ch_j', aggfunc='mean')
            avail = [r for r in region_order if r in mat.index]
            mat = mat.reindex(index=avail, columns=avail)
            matrices[cond][grp] = mat
            if not mat.isna().all().all():
                max_val = max(max_val, float(mat.abs().max().max()))
    return matrices, max_val

In [ ]:
def plot_panel_grid(matrices_c, matrices_r2, max_val_c, max_val_r2,
                    conditions, cond_labels, groups, title, filename):
    """
    Publication-ready 2x6 panel comparing coherence (Panel A) and squared
    correlation (Panel B) delta matrices.

    Parameters
    ----------
    matrices_c  : {cond: {group: DataFrame}} for coherence deltas
    matrices_r2 : {cond: {group: DataFrame}} for |r| deltas
    max_val_c   : Symmetric color bound for Panel A
    max_val_r2  : Symmetric color bound for Panel B
    title       : Figure suptitle
    filename    : Output PDF path
    """
    plt.rcParams.update({'font.size':13,'axes.titlesize':16,
                         'axes.labelsize':18,'xtick.labelsize':15,'ytick.labelsize':15})

    cmap         = LinearSegmentedColormap.from_list('custom_bwr',['#1f77b4','#ffffff','#d62728'])
    WIDTH_RATIOS = [1, 1, 1, 0.12, 1, 1, 1, 0.12]
    GS           = dict(left=0.07, right=0.97, top=0.84, bottom=0.18, hspace=0.08, wspace=0.05)

    fig      = plt.figure(figsize=(24, 13))
    gs       = fig.add_gridspec(nrows=2, ncols=8, width_ratios=WIDTH_RATIOS, **GS)
    cum      = np.cumsum([0] + WIDTH_RATIOS)
    to_fig_x = lambda x: GS['left'] + (x / cum[-1]) * (GS['right'] - GS['left'])
    first_ax = {}

    panel_configs = [
        (0, matrices_c,  max_val_c,  'A', r'Panel A: Coherence ($\mathbf{C}$)'),
        (4, matrices_r2, max_val_r2, 'B', r'Panel B: Squared Correlation ($\mathbf{\rho^2}$)'),
    ]

    LABEL_MAP = {'Exp':'EXP', 'Sham':'SHAM', 'Control':'CTRL'}

    for col_offset, matrices, vmax, letter, subtitle in panel_configs:
        for r, cond in enumerate(conditions):
            for c, grp in enumerate(groups):
                ax        = fig.add_subplot(gs[r, col_offset + c])
                left_A    = (c == 0 and letter == 'A')
                sns.heatmap(matrices[cond][grp], ax=ax, cmap=cmap,
                            center=0, vmin=-vmax, vmax=vmax,
                            annot=False, cbar=False, square=True,
                            linewidths=0.6, linecolor='lightgray',
                            xticklabels=False, yticklabels=left_A)
                if left_A:
                    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, ha='right')
                ax.tick_params(labelsize=24)
                if r == 0:
                    ax.set_title(LABEL_MAP.get(grp, grp), fontsize=25, pad=10)
                ax.set_ylabel(cond_labels[cond] if left_A else '', fontsize=32,
                              fontweight='bold', labelpad=14)
                ax.set_xlabel('')
                if r == 0 and c == 0:
                    first_ax[letter] = ax
        fig.text(to_fig_x(cum[col_offset]), 0.89, subtitle,
                 ha='left', va='bottom', fontsize=32, fontweight='bold', color='#111111')

    cbar_ax = fig.add_axes([0.25, 0.15, 0.50, 0.025])
    cbar    = fig.colorbar(first_ax['A'].collections[0], cax=cbar_ax, orientation='horizontal')
    cbar.set_label(r'$\Delta$ Connectivity (Decrease $\rightarrow$ Increase)',
                   fontsize=25, fontweight='bold', labelpad=10)
    cbar.ax.tick_params(labelsize=20)

    fig.text(GS['left'], 0.98, title, ha='left', va='top', fontsize=37, fontweight='bold')
    plt.savefig(filename, format='pdf', dpi=300, bbox_inches='tight')
    plt.show()
    plt.rcParams.update(plt.rcParamsDefault)

### 3.1 PRE → POST

In [ ]:
m_coh_pp, _ = process_delta('coherence_results_by_regions.parquet', 'coherence',
                              endpoint='POST', baseline='PRE',
                              conditions=CONDITIONS, groups=GROUPS, region_order=REGION_ORDER)
m_r2_pp,  _ = process_delta('pearson_results_by_regions.parquet', 'abs_r',
                              endpoint='POST', baseline='PRE',
                              conditions=CONDITIONS, groups=GROUPS, region_order=REGION_ORDER)
plot_panel_grid(m_coh_pp, m_r2_pp, 0.15, 0.15, CONDITIONS, COND_LABELS, GROUPS,
                title=r'Regional Connectivity Changes ($\Delta$ POST $-$ PRE)',
                filename='post_pre_delta_maps.pdf')

### 3.2 POST → FU

In [ ]:
m_coh_fu, _ = process_delta('coherence_results_by_regions.parquet', 'coherence',
                               endpoint='SEG', baseline='POST',
                               conditions=CONDITIONS, groups=GROUPS, region_order=REGION_ORDER)
m_r2_fu,  _ = process_delta('pearson_results_by_regions.parquet', 'abs_r',
                               endpoint='SEG', baseline='POST',
                               conditions=CONDITIONS, groups=GROUPS, region_order=REGION_ORDER)
plot_panel_grid(m_coh_fu, m_r2_fu, 0.15, 0.15, CONDITIONS, COND_LABELS, GROUPS,
                title=r'Regional Connectivity Changes ($\Delta$ FU $-$ POST)',
                filename='fu_post_delta_maps.pdf')